In [ ]:
# ============================================================
# silver_events_only.ipynb
#
# Builds the core silver.events table from raw play-by-play JSON.
#
# Source:
#   - dbw_hockey_lakehouse.bronze.play_by_play_raw
#
# Output:
#   - dbw_hockey_lakehouse.silver.events
#   - dbw_hockey_lakehouse.silver.events_bad_records
#   - helper views for shots / hits / penalties
#
# What this notebook is doing:
#   - parses raw play-by-play payloads
#   - explodes each game's play list into one row per event
#   - standardizes event-level columns and player-role fields
#   - applies some light DQ checks before writing to silver
#   - merges new events so reruns stay safe
#
# Write pattern:
#   - merge / upsert
#   - idempotent at the event level (game_id + event_id)
#
# Notes:
#   - this is more of a cleaning / normalization notebook than a heavy modeling one
#   - comments are a little more practical here on purpose
# ============================================================

spark.sql("CREATE SCHEMA IF NOT EXISTS dbw_hockey_lakehouse.silver")


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_PBP = "dbw_hockey_lakehouse.bronze.play_by_play_raw"
SILVER_EVENTS = "dbw_hockey_lakehouse.silver.events"
SILVER_BAD_EVENTS = "dbw_hockey_lakehouse.silver.events_bad_records"

SHOT_EVENT_TYPES = ["shot-on-goal", "missed-shot", "blocked-shot", "goal"]
PENALTY_EVENT_TYPES = ["penalty", "delayed-penalty"]
HIT_EVENT_TYPES = ["hit"]


In [ ]:
# ---------------- LOAD RAW + FILTER ALREADY-PROCESSED GAMES ----------------

bronze_df = spark.table(BRONZE_PBP)

# Silver is keyed at the event level, but filtering processed games up front
# keeps reruns lighter and avoids re-parsing a bunch of raw JSON for no reason.
if spark.catalog.tableExists(SILVER_EVENTS):
    processed_games = spark.table(SILVER_EVENTS).select("game_id").distinct()

    bronze_df = (
        bronze_df
        .join(processed_games, on="game_id", how="left_anti")
    )

# Spark-native schema inference from one non-null raw_json row.
# Basic approach, but it works fine here.
schema_row = (
    bronze_df
    .select(F.schema_of_json(F.col("raw_json")).alias("schema"))
    .where(F.col("raw_json").isNotNull())
    .limit(1)
    .collect()
)

if len(schema_row) == 0:
    print("No new games to process for silver.events.")
    parsed_df = None
else:
    schema_str = schema_row[0]["schema"]
    parsed_df = bronze_df.withColumn("pbp", F.from_json(F.col("raw_json"), schema_str))
    display(parsed_df.select("game_id", "season", "source_url").limit(10))


In [ ]:
# ---------------- EXPLODE RAW PBP INTO EVENT-LEVEL ROWS ----------------

if parsed_df is None:
    print("Skipping event extraction (no new games).")
    events_df = None
else:
    events_df = (
        parsed_df
        .select(
            F.col("game_id").cast("long").alias("game_id"),
            F.col("season").cast("long").alias("season"),
            F.col("source_url"),
            F.col("ingested_at"),
            F.col("pbp.gameDate").cast("date").alias("game_date"),
            F.col("pbp.gameType").cast("int").alias("game_type"),
            F.col("pbp.homeTeam.id").cast("long").alias("home_team_id"),
            F.col("pbp.awayTeam.id").cast("long").alias("away_team_id"),
            F.explode_outer(F.col("pbp.plays")).alias("play")
        )
    )

    display(events_df.limit(5))


In [ ]:
# ---------------- SHAPE BASE EVENT FIELDS ----------------

if events_df is None:
    print("Skipping base event shaping (no new games).")
    base = None
else:
    base = (
        events_df
        .select(
            "game_id",
            "season",
            "source_url",
            "ingested_at",
            "home_team_id",
            "away_team_id",
            F.col("play.eventId").cast("int").alias("event_id"),
            F.col("play.sortOrder").cast("int").alias("sort_order"),
            F.col("play.typeCode").cast("int").alias("type_code"),
            F.col("play.typeDescKey").cast("string").alias("event_type"),
            F.lower(F.col("play.typeDescKey")).cast("string").alias("event_type_lc"),
            F.col("play.situationCode").cast("string").alias("situation_code"),
            F.col("play.homeTeamDefendingSide").cast("string").alias("home_team_defending_side"),
            F.col("play.periodDescriptor.number").cast("int").alias("period_number"),
            F.col("play.periodDescriptor.periodType").cast("string").alias("period_type"),
            F.col("play.timeInPeriod").cast("string").alias("time_in_period"),
            F.col("play.timeRemaining").cast("string").alias("time_remaining"),
            F.col("play.coordinates.x").cast("double").alias("play_x"),
            F.col("play.coordinates.y").cast("double").alias("play_y"),
            F.col("play.details").alias("details_struct"),
            F.to_json(F.col("play.details")).alias("details_json")
        )
    )


In [ ]:
# ---------------- STANDARDIZE COMMON EVENT ATTRIBUTES ----------------

if base is None:
    print("Skipping common field cleanup (no new games).")
    with_common = None
else:
    with_common = (
        base
        # Prefer detail-level coordinates when present. Those tend to be more consistent
        # for event-specific logic. Fall back to play-level coords if needed.
        .withColumn("x", F.coalesce(F.col("details_struct.xCoord").cast("int"), F.col("play_x").cast("int")))
        .withColumn("y", F.coalesce(F.col("details_struct.yCoord").cast("int"), F.col("play_y").cast("int")))
        .withColumn("zone_code", F.col("details_struct.zoneCode").cast("string"))
        .withColumn("event_owner_team_id", F.col("details_struct.eventOwnerTeamId").cast("long"))

        # Common-ish extra fields. A lot of these are sparse by event type, which is fine.
        .withColumn("shot_type", F.col("details_struct.shotType").cast("string"))
        .withColumn("reason", F.col("details_struct.reason").cast("string"))
        .withColumn("goalie_id", F.col("details_struct.goalieInNetId").cast("long"))

        # Penalty-specific fields
        .withColumn("penalty_type_code", F.col("details_struct.typeCode").cast("string"))
        .withColumn("penalty_desc_key", F.col("details_struct.descKey").cast("string"))
        .withColumn("penalty_minutes", F.col("details_struct.duration").cast("int"))

        # Goal score snapshots are only populated on some events, so we forward-fill later.
        .withColumn("home_score_raw", F.col("details_struct.homeScore").cast("int"))
        .withColumn("away_score_raw", F.col("details_struct.awayScore").cast("int"))
    )


In [ ]:
# ---------------- STANDARDIZE PLAYER ROLES ----------------

if with_common is None:
    print("Skipping player role standardization (no new games).")
    with_roles = None
else:
    with_roles = (
        with_common
        .withColumn(
            "primary_player_id",
            F.when(F.col("event_type_lc") == "hit", F.col("details_struct.hittingPlayerId").cast("long"))
             .when(F.col("event_type_lc").isin("penalty", "delayed-penalty"), F.col("details_struct.committedByPlayerId").cast("long"))
             .when(F.col("event_type_lc").isin("shot-on-goal", "missed-shot", "blocked-shot"), F.col("details_struct.shootingPlayerId").cast("long"))
             .when(F.col("event_type_lc") == "goal", F.col("details_struct.scoringPlayerId").cast("long"))
             .when(F.col("event_type_lc").isin("giveaway", "takeaway"), F.col("details_struct.playerId").cast("long"))
             .when(F.col("event_type_lc") == "faceoff", F.col("details_struct.winningPlayerId").cast("long"))
             .otherwise(F.lit(None).cast("long"))
        )
        .withColumn(
            "secondary_player_id",
            F.when(F.col("event_type_lc") == "hit", F.col("details_struct.hitteePlayerId").cast("long"))
             .when(F.col("event_type_lc").isin("penalty", "delayed-penalty"), F.col("details_struct.drawnByPlayerId").cast("long"))
             .when(F.col("event_type_lc") == "faceoff", F.col("details_struct.losingPlayerId").cast("long"))
             .when(F.col("event_type_lc") == "blocked-shot", F.col("details_struct.blockingPlayerId").cast("long"))
             .otherwise(F.lit(None).cast("long"))
        )
    )

# This is one of the bigger normalization choices in the notebook:
# keep generic role columns in silver, then rename them semantically in gold facts.


In [ ]:
# ---------------- FILL SCORE SNAPSHOTS + DEDUP ----------------

if with_roles is None:
    print("Skipping score cleanup / dedup (no new games).")
    scored = None
else:
    score_window = Window.partitionBy("game_id").orderBy("sort_order").rowsBetween(Window.unboundedPreceding, 0)

    scored = (
        with_roles
        # Goal score is not present on every event, so carry the last known value forward.
        .withColumn("home_score", F.last("home_score_raw", ignorenulls=True).over(score_window))
        .withColumn("away_score", F.last("away_score_raw", ignorenulls=True).over(score_window))
        .drop("home_score_raw", "away_score_raw")
    )

    # Basic deduplication at the natural event grain.
    # If the raw feed repeats an event, keep the latest ingested version.
    dedup_window = Window.partitionBy("game_id", "event_id").orderBy(F.col("ingested_at").desc_nulls_last())

    scored = (
        scored
        .withColumn("_rn", F.row_number().over(dedup_window))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )


In [ ]:
# ---------------- LIGHT DATA QUALITY SPLIT ----------------

if scored is None:
    print("Skipping DQ split (no new games).")
    dq = None
    bad = None
    good = None
else:
    expects_details = F.col("event_type_lc").isin(
        "hit", "giveaway", "takeaway", "faceoff", "penalty", "delayed-penalty",
        "shot-on-goal", "blocked-shot", "missed-shot", "goal", "stoppage"
    )

    dq = (
        scored
        .withColumn(
            "dq_errors_raw",
            F.array(
                # Critical: reject the row
                F.when(F.col("game_id").isNull(), F.lit("missing_game_id").cast("string")),
                F.when(F.col("event_id").isNull(), F.lit("missing_event_id").cast("string")),
                F.when(F.col("sort_order").isNull(), F.lit("missing_sort_order").cast("string")),
                F.when(F.col("event_type").isNull(), F.lit("missing_event_type").cast("string")),

                # Warnings: keep the row, but worth surfacing
                F.when(expects_details & F.col("details_struct").isNull(), F.lit("missing_details_for_event_type").cast("string")),
                F.when(
                    F.col("event_type_lc").isin(
                        "hit", "giveaway", "takeaway", "faceoff", "penalty", "delayed-penalty",
                        "shot-on-goal", "blocked-shot", "missed-shot", "goal"
                    ) & F.col("event_owner_team_id").isNull(),
                    F.lit("missing_event_owner_team_id").cast("string")
                ),
                F.when(
                    F.col("event_type_lc").isin("shot-on-goal", "missed-shot", "goal") & F.col("goalie_id").isNull(),
                    F.lit("missing_goalie_id_on_shot_goal_event").cast("string")
                )
            )
        )
        .withColumn("dq_errors", F.expr("filter(dq_errors_raw, x -> x is not null)"))
        .drop("dq_errors_raw")
    )

    # Critical failures only. Warnings stay in the good path.
    bad = dq.where(
        F.array_contains(F.col("dq_errors"), "missing_game_id") |
        F.array_contains(F.col("dq_errors"), "missing_event_id") |
        F.array_contains(F.col("dq_errors"), "missing_sort_order") |
        F.array_contains(F.col("dq_errors"), "missing_event_type")
    )

    good = dq.where(
        ~(
            F.array_contains(F.col("dq_errors"), "missing_game_id") |
            F.array_contains(F.col("dq_errors"), "missing_event_id") |
            F.array_contains(F.col("dq_errors"), "missing_sort_order") |
            F.array_contains(F.col("dq_errors"), "missing_event_type")
        )
    )

    print("dq:", dq.count(), "bad(critical):", bad.count(), "good:", good.count())


In [ ]:
# ---------------- WRITE BAD EVENT RECORDS ----------------

if bad is None:
    print("Skipping bad-record write (no new games).")
else:
    bad_events_stage = (
        bad
        .select(
            "season",
            "game_id",
            "event_id",
            "event_type",
            "sort_order",
            "dq_errors",
            "details_json",
            "source_url",
            "ingested_at"
        )
        .dropDuplicates(["game_id", "event_id"])
    )

    bad_events_stage.createOrReplaceTempView("staging_events_bad_records")

    spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {SILVER_BAD_EVENTS} (
      season BIGINT,
      game_id BIGINT,
      event_id INT,
      event_type STRING,
      sort_order INT,
      dq_errors ARRAY<STRING>,
      details_json STRING,
      source_url STRING,
      ingested_at TIMESTAMP,
      updated_at TIMESTAMP
    )
    USING DELTA
    PARTITIONED BY (season)
    ''')

    # Merge keeps this idempotent and lets us refresh the DQ payload if the raw record changes.
    spark.sql(f'''
    MERGE INTO {SILVER_BAD_EVENTS} t
    USING staging_events_bad_records s
      ON t.game_id = s.game_id
     AND t.event_id = s.event_id
    WHEN MATCHED THEN UPDATE SET
      t.season = s.season,
      t.event_type = s.event_type,
      t.sort_order = s.sort_order,
      t.dq_errors = s.dq_errors,
      t.details_json = s.details_json,
      t.source_url = s.source_url,
      t.ingested_at = s.ingested_at,
      t.updated_at = current_timestamp()
    WHEN NOT MATCHED THEN INSERT (
      season, game_id, event_id, event_type, sort_order, dq_errors,
      details_json, source_url, ingested_at, updated_at
    ) VALUES (
      s.season, s.game_id, s.event_id, s.event_type, s.sort_order, s.dq_errors,
      s.details_json, s.source_url, s.ingested_at, current_timestamp()
    )
    ''')


In [ ]:
# ---------------- STAGE GOOD SILVER EVENTS ----------------

if good is None:
    print("Skipping silver.events stage/write (no new games).")
    events_stage = None
else:
    events_stage = (
        good
        .select(
            "season",
            "game_id",
            "event_id",
            "sort_order",
            "event_type",
            "event_type_lc",
            "type_code",
            "situation_code",
            "home_team_defending_side",
            "period_number",
            "period_type",
            "time_in_period",
            "time_remaining",
            "home_team_id",
            "away_team_id",
            "x",
            "y",
            "zone_code",
            "event_owner_team_id",
            "primary_player_id",
            "secondary_player_id",
            "goalie_id",
            "shot_type",
            "reason",
            "penalty_type_code",
            "penalty_desc_key",
            "penalty_minutes",
            "home_score",
            "away_score",
            "details_json",
            "source_url",
            "ingested_at"
        )
    )

    events_stage.createOrReplaceTempView("staging_silver_events")
    print("events_stage rows:", events_stage.count())


In [ ]:
# ---------------- CREATE TARGET TABLE + MERGE ----------------

if events_stage is None:
    print("Skipping silver.events merge (no new games).")
else:
    spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {SILVER_EVENTS} (
      season BIGINT,
      game_id BIGINT,
      event_id INT,
      sort_order INT,
      event_type STRING,
      event_type_lc STRING,
      type_code INT,
      situation_code STRING,
      home_team_defending_side STRING,
      period_number INT,
      period_type STRING,
      time_in_period STRING,
      time_remaining STRING,
      home_team_id BIGINT,
      away_team_id BIGINT,
      x INT,
      y INT,
      zone_code STRING,
      event_owner_team_id BIGINT,
      primary_player_id BIGINT,
      secondary_player_id BIGINT,
      goalie_id BIGINT,
      shot_type STRING,
      reason STRING,
      penalty_type_code STRING,
      penalty_desc_key STRING,
      penalty_minutes INT,
      home_score INT,
      away_score INT,
      details_json STRING,
      source_url STRING,
      ingested_at TIMESTAMP
    )
    USING DELTA
    ''')

    # Small schema guard for older versions of silver.events.
    # If the table already exists from an earlier run, add event_type_lc
    # before the merge so the update/insert logic does not fail.
    target_table = SILVER_EVENTS

    if spark.catalog.tableExists(target_table):
        existing_cols = {field.name for field in spark.table(target_table).schema.fields}

        if "event_type_lc" not in existing_cols:
            spark.sql(f"""
            ALTER TABLE {target_table}
            ADD COLUMNS (event_type_lc STRING)
            """)

    # Merge on the natural event key. This keeps reruns safe and also lets
    # corrected source values flow through without needing a full overwrite.
    spark.sql(f'''
    MERGE INTO {SILVER_EVENTS} t
    USING staging_silver_events s
      ON t.game_id = s.game_id
     AND t.event_id = s.event_id
    WHEN MATCHED THEN UPDATE SET
      t.season = s.season,
      t.sort_order = s.sort_order,
      t.event_type = s.event_type,
      t.event_type_lc = s.event_type_lc,
      t.type_code = s.type_code,
      t.situation_code = s.situation_code,
      t.home_team_defending_side = s.home_team_defending_side,
      t.period_number = s.period_number,
      t.period_type = s.period_type,
      t.time_in_period = s.time_in_period,
      t.time_remaining = s.time_remaining,
      t.home_team_id = s.home_team_id,
      t.away_team_id = s.away_team_id,
      t.x = s.x,
      t.y = s.y,
      t.zone_code = s.zone_code,
      t.event_owner_team_id = s.event_owner_team_id,
      t.primary_player_id = s.primary_player_id,
      t.secondary_player_id = s.secondary_player_id,
      t.goalie_id = s.goalie_id,
      t.shot_type = s.shot_type,
      t.reason = s.reason,
      t.penalty_type_code = s.penalty_type_code,
      t.penalty_desc_key = s.penalty_desc_key,
      t.penalty_minutes = s.penalty_minutes,
      t.home_score = s.home_score,
      t.away_score = s.away_score,
      t.details_json = s.details_json,
      t.source_url = s.source_url,
      t.ingested_at = s.ingested_at
    WHEN NOT MATCHED THEN INSERT (
      season, game_id, event_id, sort_order, event_type, event_type_lc, type_code,
      situation_code, home_team_defending_side, period_number, period_type,
      time_in_period, time_remaining, home_team_id, away_team_id, x, y, zone_code,
      event_owner_team_id, primary_player_id, secondary_player_id, goalie_id,
      shot_type, reason, penalty_type_code, penalty_desc_key, penalty_minutes,
      home_score, away_score, details_json, source_url, ingested_at
    ) VALUES (
      s.season, s.game_id, s.event_id, s.sort_order, s.event_type, s.event_type_lc, s.type_code,
      s.situation_code, s.home_team_defending_side, s.period_number, s.period_type,
      s.time_in_period, s.time_remaining, s.home_team_id, s.away_team_id, s.x, s.y, s.zone_code,
      s.event_owner_team_id, s.primary_player_id, s.secondary_player_id, s.goalie_id,
      s.shot_type, s.reason, s.penalty_type_code, s.penalty_desc_key, s.penalty_minutes,
      s.home_score, s.away_score, s.details_json, s.source_url, s.ingested_at
    )
    ''')

In [ ]:
# ---------------- HELPER VIEWS ----------------

# These are just convenience views for downstream notebooks.
# Nothing fancy, just making common event slices easier to grab.

spark.sql(f'''
CREATE OR REPLACE VIEW dbw_hockey_lakehouse.silver.v_shots AS
SELECT *
FROM {SILVER_EVENTS}
WHERE event_type_lc IN ('shot-on-goal', 'missed-shot', 'blocked-shot', 'goal')
''')

spark.sql(f'''
CREATE OR REPLACE VIEW dbw_hockey_lakehouse.silver.v_hits AS
SELECT *
FROM {SILVER_EVENTS}
WHERE event_type_lc = 'hit'
''')

spark.sql(f'''
CREATE OR REPLACE VIEW dbw_hockey_lakehouse.silver.v_penalties AS
SELECT *
FROM {SILVER_EVENTS}
WHERE event_type_lc IN ('penalty', 'delayed-penalty')
''')


In [ ]:
# ---------------- QUICK SANITY CHECKS ----------------

if spark.catalog.tableExists(SILVER_EVENTS):
    display(
        spark.table(SILVER_EVENTS)
        .groupBy("event_type")
        .count()
        .orderBy(F.col("count").desc())
    )

if spark.catalog.tableExists(SILVER_BAD_EVENTS):
    display(
        spark.table(SILVER_BAD_EVENTS)
        .orderBy(F.col("ingested_at").desc_nulls_last())
        .limit(20)
    )
